# 07 · Enderezado de la rotación: ¿H6 se recupera de un paciente girado?

Corrección pedida por el profesor: normalizar la rotación con los metadatos (productos punto y cruz,
artículo de Ferro Sánchez et al., MethodsX 2024) y corregir el giro del paciente.
Este notebook **no entrena**: evalúa el modelo H6 ya entrenado en los **15 casos de validación** (test no se toca).

| Corrida | Qué mide |
|---|---|
| `base` | H6 sin giro (referencia) |
| `rotmm30` / `rotmm30_norm` | paciente girado 30° en milímetros, sin y con enderezado |
| `rotmm60` / `rotmm60_norm` | paciente girado 60° en milímetros, sin y con enderezado |

**Antes de correr:** activar GPU (Colab: Entorno de ejecución → Cambiar tipo → T4 · Kaggle: Settings → Accelerator → GPU) y llenar la celda de configuración.
Tiempo aproximado con GPU: 15–25 min.

In [1]:
!nvidia-smi -L || echo "SIN GPU: activa la GPU antes de seguir"

GPU 0: Tesla T4 (UUID: GPU-1c9277f0-ce6c-642c-3fa2-ff0095790ef2)


## 1. Configuración (lo único que hay que editar)

Se necesitan dos cosas además del repo:

1. **El checkpoint de H6**: `h6_auggeo_rot35_canon_ema_w32_e60_s42.pth`.
2. **Los datos de validación**, de una de estas dos formas:
   - **Opción A (recomendada, más rápida):** `cache_s10_val.zip`, el caché ya preparado de los 15 casos de validación (lo comparte María).
   - **Opción B:** la carpeta `data/raw` original de PENGWIN (las 3 carpetas `PENGWIN_CT_train_*`); el notebook construye el caché.

En **Colab** súbanlos a una carpeta de Drive; en **Kaggle**, como un *Dataset* privado agregado al notebook (aparece en `/kaggle/input/...`; Kaggle descomprime el zip automáticamente y el notebook lo maneja).

In [4]:
import os
EN_KAGGLE = os.path.exists("/kaggle/working")

REPO = "pengwin-analitica-2026/proyecto2-analitica-datos"
RAMA = "feat/normalizacion-orientacion"
GITHUB_TOKEN = ""
REPO_ZIP = ""

if EN_KAGGLE:
    CKPT      = "/kaggle/input/pengwin-h6/h6_auggeo_rot35_canon_ema_w32_e60_s42.pth"
    CACHE_ZIP = "/kaggle/input/pengwin-h6"
    REPO_ZIP  = "/kaggle/input/pengwin-h6/codigo_pengwin.zip"
    RAW_DIR   = ""
    TRABAJO   = "/kaggle/working"
else:
    from google.colab import drive
    drive.mount("/content/drive")
    CKPT      = "/content/drive/MyDrive/PENGWIN/h6_auggeo_rot35_canon_ema_w32_e60_s42.pth"
    CACHE_ZIP = "/content/drive/MyDrive/PENGWIN/cache_s10_val.zip"
    REPO_ZIP  = "/content/drive/MyDrive/PENGWIN/codigo_pengwin.zip"
    RAW_DIR   = ""
    TRABAJO   = "/content"

for nombre, ruta in [("checkpoint", CKPT), ("caché", CACHE_ZIP), ("código", REPO_ZIP)]:
    print(f"{nombre:12s} {'OK ' if os.path.exists(ruta) else 'NO EXISTE'}  {ruta}")
assert os.path.exists(CKPT), "No encuentro el checkpoint"
assert os.path.exists(CACHE_ZIP), "No encuentro el caché"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
checkpoint   OK   /content/drive/MyDrive/PENGWIN/h6_auggeo_rot35_canon_ema_w32_e60_s42.pth
caché        OK   /content/drive/MyDrive/PENGWIN/cache_s10_val.zip
código       OK   /content/drive/MyDrive/PENGWIN/codigo_pengwin.zip


## 2. Código del repo (rama de la corrección)

In [5]:
import os, glob, zipfile, shutil
os.chdir(TRABAJO)
if not os.path.exists("repo"):
    if REPO_ZIP and os.path.isfile(REPO_ZIP):
        with zipfile.ZipFile(REPO_ZIP) as z:
            z.extractall("repo_zip")
        carpeta = glob.glob("repo_zip/*")[0]
        shutil.move(carpeta, "repo")
    elif REPO_ZIP and os.path.isdir(REPO_ZIP):
        # Kaggle ya descomprimió el zip
        src = glob.glob(os.path.join(os.path.dirname(REPO_ZIP), "**", "src", "normalizacion.py"), recursive=True)
        assert src, "No encontré el código (src/normalizacion.py)"
        shutil.copytree(os.path.dirname(os.path.dirname(src[0])), "repo")
    else:
        raise SystemExit("No encuentro codigo_pengwin.zip: revisa REPO_ZIP en la celda de configuración")
os.chdir("repo")
assert os.path.exists("src/normalizacion.py"), "El código no es de la rama feat/normalizacion-orientacion"
print("Código listo en", os.getcwd())
!pip -q install SimpleITK scikit-image

Código listo en /content/repo
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 23.1 MB/s eta 0:00:00


## 3. Datos de validación

In [7]:
import os, shutil
os.makedirs("outputs/s10", exist_ok=True)
shutil.copy(CKPT, "outputs/s10/")
CK = "outputs/s10/" + os.path.basename(CKPT)

if CACHE_ZIP and os.path.isdir(CACHE_ZIP):
    # Kaggle ya descomprimió el zip: buscar index.csv y copiar su carpeta (s10) a data/cache/s10
    import glob
    indices = glob.glob(os.path.join(CACHE_ZIP, "**", "index.csv"), recursive=True)
    assert indices, f"No encontré index.csv dentro de {CACHE_ZIP}"
    os.makedirs("data/cache", exist_ok=True)
    shutil.copytree(os.path.dirname(indices[0]), "data/cache/s10", dirs_exist_ok=True)
elif CACHE_ZIP:
    # El zip trae data/cache/s10/index.csv y data/cache/s10/cache_384/*.npz
    !unzip -q -o "{CACHE_ZIP}" -d .
else:
    os.makedirs("data", exist_ok=True)
    if not os.path.exists("data/raw"):
        os.symlink(RAW_DIR, "data/raw")
    VAL = "014 019 024 027 033 044 050 057 060 067 077 078 087 095 099"
    !python src/build_case_cache.py --out-dir data/cache/s10 --sizes 384 --cases {VAL}

CD = "data/cache/s10/cache_384"
print(len([f for f in os.listdir(CD) if f.endswith(".npz")]), "casos en el caché (deben ser 15)")

15 casos en el caché (deben ser 15)


## 4. Pruebas automáticas (segundos)

In [8]:
!python -m unittest tests.test_normalizacion.TestPaso1Metadatos tests.test_normalizacion.TestPaso2GiroAxial tests.test_normalizacion.TestPaso2EnCache tests.test_evaluar_normalizado tests.test_aumento_fisico 2>&1 | tail -3

Ran 27 tests in 12.097s

OK


## 5. Las 5 evaluaciones (validación, con TTA)

`rotmm=X` gira al paciente X grados **en milímetros reales** (el `rot=X` anterior giraba en píxeles y, en 10 de los 15 casos,
que tienen píxeles no cuadrados, deformaba la pelvis). `--normalizar` endereza cada volumen antes del modelo.

In [9]:
CORRIDAS = [("base", ""),
            ("rotmm30", "--stress rotmm=30"), ("rotmm30_norm", "--stress rotmm=30 --normalizar"),
            ("rotmm60", "--stress rotmm=60"), ("rotmm60_norm", "--stress rotmm=60 --normalizar")]
os.makedirs("outputs/normalizacion", exist_ok=True)
for nombre, extra in CORRIDAS:
    print(f"\n========== {nombre} ==========")
    !python src/evaluate_volumes.py --checkpoint {CK} --cache-dir {CD} --split val --tta {extra} --output outputs/normalizacion/h6_{nombre}.json 2>&1 | grep -E "casos de|caso |giro estimado|Guardado"


========== base ==========
15 casos de val | dispositivo cuda | TTA=True | segundo pase=False | normalizar=False
  1/15 caso 014: 298 cortes (49 s)
  2/15 caso 019: 350 cortes (102 s)
  3/15 caso 024: 350 cortes (155 s)
  4/15 caso 027: 313 cortes (202 s)
  5/15 caso 033: 365 cortes (259 s)
  6/15 caso 044: 409 cortes (326 s)
  7/15 caso 050: 326 cortes (378 s)
  8/15 caso 057: 341 cortes (432 s)
  9/15 caso 060: 333 cortes (484 s)
  10/15 caso 067: 369 cortes (542 s)
  11/15 caso 077: 350 cortes (594 s)
  12/15 caso 078: 388 cortes (655 s)
  13/15 caso 087: 312 cortes (704 s)
  14/15 caso 095: 257 cortes (743 s)
  15/15 caso 099: 286 cortes (786 s)
Guardado: outputs/normalizacion/h6_base.json

========== rotmm30 ==========
15 casos de val | dispositivo cuda | TTA=True | segundo pase=False | normalizar=False
  1/15 caso 014: 298 cortes (51 s)
  2/15 caso 019: 350 cortes (108 s)
  3/15 caso 024: 350 cortes (165 s)
  4/15 caso 027: 313 cortes (216 s)
  5/15 caso 033: 365 cortes (276 s)


## 6. Tabla de resultados

In [10]:
import json
import pandas as pd
METRICAS = ["classification_f1_macro", "classification_auc_macro", "detection_map50", "detection_iou_mean",
            "segmentation_dice_macro", "instance_dice_matched"]
filas = []
for nombre, _ in CORRIDAS:
    r = json.load(open(f"outputs/normalizacion/h6_{nombre}.json"))
    fila = {"corrida": nombre, **{m: round(r["subsets"]["todo"].get(m, float("nan")), 3) for m in METRICAS}}
    giros = r.get("giros", {})
    fila["casos_enderezados"] = sum(g["corregido"] for g in giros.values()) if giros else "-"
    filas.append(fila)
tabla = pd.DataFrame(filas).set_index("corrida")
tabla.to_csv("outputs/normalizacion/resumen_h6_normalizacion.csv")
tabla

,classification_f1_macro,classification_auc_macro,detection_map50,detection_iou_mean,segmentation_dice_macro,instance_dice_matched,casos_enderezados
corrida,,,,,,,
base,0.992,0.999,0.971,0.949,0.971,0.869,-
rotmm30,0.990,0.999,0.943,0.927,0.961,0.853,-
rotmm30_norm,0.992,0.999,0.971,0.945,0.959,0.854,15
rotmm60,0.959,0.988,0.311,0.392,0.715,0.608,-
rotmm60_norm,0.989,0.999,0.963,0.928,0.957,0.851,15


In [11]:
# Ángulo estimado por caso en las corridas con enderezado (lo ideal: ≈ giro de estrés + giro propio del paciente)
for nombre in ("rotmm30_norm", "rotmm60_norm"):
    g = json.load(open(f"outputs/normalizacion/h6_{nombre}.json"))["giros"]
    print(nombre, {c: v["angulo"] for c, v in g.items()})

rotmm30_norm {'014': 36.25, '019': 31.5, '024': 28.25, '027': 33.5, '033': 31.0, '044': 18.5, '050': 27.0, '057': 34.0, '060': 31.5, '067': 29.25, '077': 33.25, '078': 32.0, '087': 25.25, '095': 24.0, '099': 29.0}
rotmm60_norm {'014': 66.25, '019': 61.5, '024': 58.25, '027': 63.5, '033': 59.0, '044': 39.25, '050': 53.5, '057': 63.75, '060': 62.0, '067': 58.25, '077': 63.0, '078': 58.0, '087': 54.75, '095': 54.0, '099': 52.5}


## 7. Descargar resultados (enviárselos a María)

In [12]:
!cd outputs && zip -q -r ../resultados_normalizacion.zip normalizacion
print("Archivo listo:", os.path.abspath("resultados_normalizacion.zip"), "(en Kaggle: pestaña Output)")
if not EN_KAGGLE:
    from google.colab import files
    files.download("resultados_normalizacion.zip")

Archivo listo: /content/repo/resultados_normalizacion.zip (en Kaggle: pestaña Output)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>